# StoreSense: Exploratory Data Analysis

**Owner:** Member 1 (Data & ETL Lead)

A first look at the warehouse built by `python etl/load.py`. Every number and chart below is
computed live from `data/warehouse/storesense.duckdb`, so run the ETL first and then
**Run All**.

What each section helps us decide:

| Section | Why it matters later |
|---|---|
| Warehouse size | confirms the load worked and shows how big mining inputs can get |
| Orders per user | the spread of *frequency* for customer segmentation |
| When people order | what the OLAP slice / dice / pivot demos should show |
| Days since prior order | the spike at 30 is a cap in the data, not real behaviour |
| Basket size | the *monetary proxy* (there are no prices) and rule-mining basket sizes |
| Departments and products | where the volume is, and which products reorder |
| Data quality notes | the data caveats to state in the report |

In [ ]:
import pandas as pd
import plotly.express as px

from common.db import query_df

pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Warehouse size

In [ ]:
tables = ["dim_time", "dim_product", "dim_user", "fact_order", "fact_order_line"]
sizes = pd.DataFrame(
    {"table": tables,
     "rows": [query_df(f"SELECT COUNT(*) AS n FROM {t}")["n"][0] for t in tables]}
)
sizes

## 2. How many orders do users place?

`total_orders` per user (from `dim_user`). The dataset only includes users with at least 4 orders.

In [ ]:
users = query_df("SELECT total_orders, avg_basket_size, reorder_ratio FROM dim_user")
print(users["total_orders"].describe().round(2).to_string())
px.histogram(users, x="total_orders", nbins=60, title="Orders per user")

## 3. When do people order?

Day-of-week coding is undocumented in the dataset; we **assume 0 = Sunday** (see `etl/transform.py`).

In [ ]:
by_slot = query_df('''
    SELECT t.order_dow, t.day_name, t.order_hour_of_day, COUNT(*) AS orders
    FROM fact_order o
    JOIN dim_time t ON o.time_key = t.time_key
    GROUP BY t.order_dow, t.day_name, t.order_hour_of_day
''')

by_day = by_slot.groupby(["order_dow", "day_name"], as_index=False)["orders"].sum().sort_values("order_dow")
px.bar(by_day, x="day_name", y="orders", title="Orders by day of week")

In [ ]:
by_hour = by_slot.groupby("order_hour_of_day", as_index=False)["orders"].sum()
px.bar(by_hour, x="order_hour_of_day", y="orders", title="Orders by hour of day")

In [ ]:
heat = by_slot.pivot(index="day_name", columns="order_hour_of_day", values="orders")
day_order = by_day["day_name"].tolist()
px.imshow(heat.loc[day_order], aspect="auto", title="Orders: day of week x hour of day",
          labels={"x": "hour of day", "y": "day", "color": "orders"})

## 4. Days since the previous order

`days_since_prior` is **NULL on every user's first order** (there is no earlier order) and
**capped at 30** in the source data. A tall bar at 30 therefore means "30 days or more",
not "exactly 30".

In [ ]:
gaps = query_df('''
    SELECT days_since_prior, COUNT(*) AS orders
    FROM fact_order
    WHERE days_since_prior IS NOT NULL
    GROUP BY days_since_prior
    ORDER BY days_since_prior
''')
px.bar(gaps, x="days_since_prior", y="orders", title="Days since prior order (30 = capped)")

## 5. Basket size

Items per order. With no prices in the data, this is our stand-in for the "monetary" part of RFM.
Test orders have `basket_size = 0` (no products), so they are excluded here.

In [ ]:
baskets = query_df('''
    SELECT basket_size, COUNT(*) AS orders
    FROM fact_order
    WHERE basket_size > 0
    GROUP BY basket_size
    ORDER BY basket_size
''')
stats = query_df('''
    SELECT AVG(basket_size) AS mean, MEDIAN(basket_size) AS median,
           QUANTILE_CONT(basket_size, 0.95) AS p95, MAX(basket_size) AS max
    FROM fact_order WHERE basket_size > 0
''')
print(stats.round(2).to_string(index=False))
px.bar(baskets[baskets["basket_size"] <= 60], x="basket_size", y="orders",
       title="Basket size (orders with up to 60 items)")

## 6. Departments and products

In [ ]:
depts = query_df('''
    SELECT p.department,
           COUNT(*)        AS order_lines,
           AVG(f.reordered) AS reorder_rate
    FROM fact_order_line f
    JOIN dim_product p ON f.product_key = p.product_key
    GROUP BY p.department
    ORDER BY order_lines DESC
''')
px.bar(depts, x="department", y="order_lines", title="Order lines by department")

In [ ]:
px.bar(depts.sort_values("reorder_rate", ascending=False), x="department", y="reorder_rate",
       title="Reorder rate by department")

In [ ]:
top_products = query_df('''
    SELECT p.product_name, p.aisle, p.department, COUNT(*) AS order_lines, AVG(f.reordered) AS reorder_rate
    FROM fact_order_line f
    JOIN dim_product p ON f.product_key = p.product_key
    GROUP BY p.product_name, p.aisle, p.department
    ORDER BY order_lines DESC
    LIMIT 15
''')
top_products

## 7. Data quality notes

Facts to state openly in the report (the brief asks for this).

In [ ]:
quality = {
    "orders with no products (test orders), kept with basket_size = 0":
        query_df("SELECT COUNT(*) AS n FROM fact_order WHERE basket_size = 0")["n"][0],
    "first orders (days_since_prior is NULL by design)":
        query_df("SELECT COUNT(*) AS n FROM fact_order WHERE days_since_prior IS NULL")["n"][0],
    "orders at the 30-day cap":
        query_df("SELECT COUNT(*) AS n FROM fact_order WHERE days_since_prior = 30")["n"][0],
    "products in the 'missing' department":
        query_df("SELECT COUNT(*) AS n FROM dim_product WHERE department = 'missing'")["n"][0],
    "products in the 'missing' aisle":
        query_df("SELECT COUNT(*) AS n FROM dim_product WHERE aisle = 'missing'")["n"][0],
    "products in the 'other' department":
        query_df("SELECT COUNT(*) AS n FROM dim_product WHERE department = 'other'")["n"][0],
}
pd.Series(quality, name="count").to_frame()